# RQ3: Operational Complexity Analysis

**Research Question:** What is the deployment complexity (configuration LOC, deployment time) for each architecture?

## Hypotheses Tested

| ID | Statement | Testable Prediction |
|---|---|---|
| H3a | Triton requires fewest lines of application code | triton.application_code_loc < monolithic.application_code_loc |
| H3b | Microservices requires most total configuration | microservices.total_config_loc > max(monolithic, triton) |
| H3c | Monolithic has shortest deployment time | monolithic.deployment_time < min(microservices, triton) |

In [ ]:
import sys
from pathlib import Path

# Get project root (works regardless of current working directory)
_notebook_dir = Path().resolve()
if _notebook_dir.name == 'notebooks' and _notebook_dir.parent.name == 'analysis':
    _project_root = _notebook_dir.parent.parent
else:
    _project_root = _notebook_dir

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from analysis.utilities.loaders import ResultsLoader

# Load configuration from experiment.yaml
ResultsLoader._load_config_from_yaml()

# Get standard constants from ResultsLoader
ARCH_COLORS = ResultsLoader.ARCH_COLORS
ARCH_DISPLAY_NAMES = ResultsLoader.ARCH_DISPLAY_NAMES

# Cohen's d effect size calculation
def cohens_d(group1, group2):
    """Calculate Cohen's d effect size between two groups.
    
    Cohen's d measures the standardized difference between two means.
    Interpretation: |d| < 0.2 negligible, 0.2-0.5 small, 0.5-0.8 medium, > 0.8 large
    """
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    # Pooled standard deviation
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (group1.mean() - group2.mean()) / pooled_std if pooled_std > 0 else 0

def interp_d(d):
    """Interpret Cohen's d effect size magnitude."""
    d_abs = abs(d) if d is not None else 0
    if d_abs < 0.2:
        return "negligible"
    elif d_abs < 0.5:
        return "small"
    elif d_abs < 0.8:
        return "medium"
    else:
        return "large"

# Set publication-quality defaults with sans-serif fonts
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.size': 11,
    'font.family': 'sans-serif',
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

# Output directory for plots
PLOTS_DIR = Path('../plots/rq3')
PLOTS_DIR.mkdir(exist_ok=True, parents=True)

print(f"Plots will be saved to: {PLOTS_DIR.resolve()}")

In [ ]:
# Data loading with fail-fast validation
# All RQ3 data (generated and static) lives in resources/
# Metrics data (LOC, deployment times) in results/metrics

METRICS_DIR = _project_root / 'results' / 'metrics'
# Pricing data in analysis/data
PRICING_DIR = _project_root / 'analysis' / 'data'

# Load LOC counts
loc_path = METRICS_DIR / 'loc_counts.csv'
assert loc_path.exists(), f"loc_counts.csv not found at {loc_path.resolve()}"
loc_counts = pd.read_csv(loc_path)

# Load deployment times
deploy_path = METRICS_DIR / 'deployment_times.csv'
assert deploy_path.exists(), f"deployment_times.csv not found at {deploy_path.resolve()}"
deployment_times = pd.read_csv(deploy_path)

# Load RQ3 summary
summary_path = METRICS_DIR / 'metrics_summary.csv'
assert summary_path.exists(), f"metrics_summary.csv not found at {summary_path.resolve()}"
rq3_summary = pd.read_csv(summary_path)

# Pricing data paths (parsed in cost analysis section)
aws_path = PRICING_DIR / 'aws_pricing.csv'
assert aws_path.exists(), f"aws_pricing.csv not found at {aws_path.resolve()}"

gcp_path = PRICING_DIR / 'gcp_pricing.csv'
assert gcp_path.exists(), f"gcp_pricing.csv not found at {gcp_path.resolve()}"

# Data overview
print("=" * 60)
print("DATA OVERVIEW")
print("=" * 60)
print(f"\nArchitectures: {loc_counts['architecture'].unique().tolist()}")
print(f"LOC categories: {loc_counts['category'].unique().tolist()}")
print(f"Deployment runs per architecture: {deployment_times.groupby('architecture').size().to_dict()}")

print("\n--- LOC Counts ---")
display(loc_counts)

print("\n--- Deployment Times ---")
display(deployment_times.head())

## Data Validation

In [ ]:
# Data validation assertions
expected_architectures = {'monolithic', 'microservices', 'triton'}
actual_architectures = set(loc_counts['architecture'].unique())
assert actual_architectures == expected_architectures, \
    f"Expected 3 architectures {expected_architectures}, got {actual_architectures}"

# Validate LOC categories
expected_categories = {'application', 'configuration'}
actual_categories = set(loc_counts['category'].unique())
assert actual_categories == expected_categories, \
    f"Expected categories {expected_categories}, got {actual_categories}"

# Validate deployment time runs (3 per architecture)
for arch in expected_architectures:
    runs = len(deployment_times[deployment_times['architecture'] == arch])
    assert runs == 3, f"Expected 3 runs for {arch}, got {runs}"

print("Basic data validation passed")
print(f"  - 3 architectures: {sorted(actual_architectures)}")
print(f"  - 2 LOC categories: {sorted(actual_categories)}")
print(f"  - 3 deployment runs per architecture")

# Extended Data Validation
print()
print("Extended data validation:")

# LOC validation - reasonable ranges
assert loc_counts['loc'].min() > 0, "Non-positive LOC detected"
total_loc = loc_counts.groupby('architecture')['loc'].sum()
assert total_loc.min() > 100, f"Suspiciously low total LOC: {total_loc.min()}"
assert total_loc.max() < 2000, f"Suspiciously high total LOC: {total_loc.max()}"

# Deployment time validation
assert deployment_times['total_time_seconds'].min() > 0, "Non-positive deployment time"
deploy_means = deployment_times.groupby('architecture')['total_time_seconds'].mean()

# Cross-check against known baseline (from prior runs)
# Monolithic should be fastest (~50-70s), Triton slowest (~500-700s)
assert deploy_means['monolithic'] < 100, f"Monolithic too slow: {deploy_means['monolithic']:.0f}s"
assert deploy_means['triton'] > 400, f"Triton unexpectedly fast: {deploy_means['triton']:.0f}s"

print("Extended data validation passed")
print(f"  - LOC range: {total_loc.min():.0f} - {total_loc.max():.0f}")
print(f"  - Deployment times: mono={deploy_means['monolithic']:.0f}s, triton={deploy_means['triton']:.0f}s")

## 1. Lines of Code Analysis (H3a, H3b)

Compares application code and configuration LOC across architectures.

In [ ]:
# LOC stacked horizontal bar chart
# Pivot data: architecture as index, category as columns
loc_pivot = loc_counts.pivot(index='architecture', columns='category', values='loc')

# Reorder to match display order (monolithic, microservices, triton)
arch_order = ['monolithic', 'microservices', 'triton']
loc_pivot = loc_pivot.reindex(arch_order)

# Create figure
fig, ax = plt.subplots(figsize=(10, 5))

# Create stacked horizontal bar chart
y_pos = np.arange(len(arch_order))
bar_height = 0.6

# Application code bars (primary color)
app_bars = ax.barh(y_pos, loc_pivot['application'], bar_height, 
                   label='Application Code',
                   color=[ARCH_COLORS[a] for a in arch_order])

# Configuration bars (lighter shade, stacked)
config_bars = ax.barh(y_pos, loc_pivot['configuration'], bar_height,
                      left=loc_pivot['application'],
                      label='Configuration',
                      color=[ARCH_COLORS[a] for a in arch_order],
                      alpha=0.5,
                      hatch='//')

# Add value labels on bars
for i, arch in enumerate(arch_order):
    app_loc = loc_pivot.loc[arch, 'application']
    config_loc = loc_pivot.loc[arch, 'configuration']
    
    # Application LOC label (centered in app bar)
    ax.text(app_loc / 2, i, f'{int(app_loc)}', 
            ha='center', va='center', fontweight='bold', color='white')
    
    # Configuration LOC label (centered in config bar)
    ax.text(app_loc + config_loc / 2, i, f'{int(config_loc)}', 
            ha='center', va='center', fontweight='bold')
    
    # Total label at end
    total = app_loc + config_loc
    ax.text(total + 10, i, f'Total: {int(total)}', 
            ha='left', va='center', fontsize=10)

# Customize axes
ax.set_yticks(y_pos)
ax.set_yticklabels([ARCH_DISPLAY_NAMES[a] for a in arch_order])
ax.set_xlabel('Lines of Code')
ax.set_title('Lines of Code by Architecture and Category')
ax.legend(loc='lower right')
ax.set_xlim(0, loc_pivot.sum(axis=1).max() * 1.2)

plt.tight_layout()

# Save in both formats
plt.savefig(PLOTS_DIR / 'rq3_loc_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: rq3_loc_comparison.png")

### H3a: Triton requires fewest application LOC

In [ ]:
# H3a hypothesis validation
# Extract application LOC for each architecture
app_loc = loc_counts[loc_counts['category'] == 'application'].set_index('architecture')['loc']

triton_app_loc = app_loc['triton']
monolithic_app_loc = app_loc['monolithic']
microservices_app_loc = app_loc['microservices']

print("H3a: Application Code LOC Comparison")
print("=" * 50)
print(f"\n{'Architecture':<15} {'Application LOC':>15}")
print("-" * 32)
for arch in ['monolithic', 'microservices', 'triton']:
    print(f"{ARCH_DISPLAY_NAMES[arch]:<15} {app_loc[arch]:>15}")

print()

# Testable prediction: triton.application_code_loc < monolithic.application_code_loc
h3a_supported = triton_app_loc < monolithic_app_loc

print(f"Testable Prediction: triton.app_loc ({triton_app_loc}) < monolithic.app_loc ({monolithic_app_loc})")
print(f"Result: {triton_app_loc} < {monolithic_app_loc} = {h3a_supported}")
print(f"\nH3a Supported: {h3a_supported}")

if not h3a_supported:
    print(f"\nNote: Triton ({triton_app_loc} LOC) actually requires MORE application code than ")
    print(f"      Monolithic ({monolithic_app_loc} LOC), likely due to gateway service code.")

### H3b: Microservices requires most configuration LOC

In [ ]:
# H3b hypothesis validation
# Extract configuration LOC for each architecture
config_loc = loc_counts[loc_counts['category'] == 'configuration'].set_index('architecture')['loc']

microservices_config_loc = config_loc['microservices']
monolithic_config_loc = config_loc['monolithic']
triton_config_loc = config_loc['triton']

max_others = max(monolithic_config_loc, triton_config_loc)

print("H3b: Configuration LOC Comparison")
print("=" * 50)
print(f"\n{'Architecture':<15} {'Configuration LOC':>17}")
print("-" * 34)
for arch in ['monolithic', 'microservices', 'triton']:
    print(f"{ARCH_DISPLAY_NAMES[arch]:<15} {config_loc[arch]:>17}")

print()

# Testable prediction: microservices.total_config_loc > max(monolithic, triton)
h3b_supported = microservices_config_loc > max_others

print(f"Testable Prediction: microservices.config_loc ({microservices_config_loc}) > max(monolithic, triton) ({max_others})")
print(f"Result: {microservices_config_loc} > {max_others} = {h3b_supported}")
print(f"\nH3b Supported: {h3b_supported}")

In [ ]:
# Total LOC summary table
loc_summary = loc_pivot.copy()
loc_summary.columns = ['App LOC', 'Config LOC']
loc_summary['Total LOC'] = loc_summary['App LOC'] + loc_summary['Config LOC']

# Add ratios vs monolithic baseline
mono_total = loc_summary.loc['monolithic', 'Total LOC']
loc_summary['Ratio vs Mono'] = (loc_summary['Total LOC'] / mono_total).round(2)

# Rename index for display
loc_summary.index = [ARCH_DISPLAY_NAMES[a] for a in loc_summary.index]

print("\nTotal LOC Summary")
print("=" * 60)
display(loc_summary)

# Save to CSV
loc_summary.to_csv(PLOTS_DIR / 'rq3_loc_summary.csv')
print("\nSaved: rq3_loc_summary.csv")

## 2. Deployment Time Analysis (H3c)

In [ ]:
# Deployment time visualization - bar chart with error bars
# Aggregate by architecture: mean and std
deploy_agg = deployment_times.groupby('architecture')['total_time_seconds'].agg(['mean', 'std']).reset_index()
deploy_agg = deploy_agg.set_index('architecture').reindex(arch_order).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))

# Create bar chart
x_pos = np.arange(len(arch_order))
bars = ax.bar(x_pos, deploy_agg['mean'], 
              yerr=deploy_agg['std'],
              capsize=5,
              color=[ARCH_COLORS[a] for a in arch_order],
              edgecolor='black',
              linewidth=1.2)

# Add value labels on bars
for i, (arch, row) in enumerate(zip(arch_order, deploy_agg.itertuples())):
    ax.text(i, row.mean + row.std + 20, f'{row.mean:.1f}s', 
            ha='center', va='bottom', fontweight='bold')

# Customize axes
ax.set_xticks(x_pos)
ax.set_xticklabels([ARCH_DISPLAY_NAMES[a] for a in arch_order])
ax.set_ylabel('Time (seconds)')
ax.set_title('Deployment Time by Architecture')
ax.set_ylim(0, deploy_agg['mean'].max() * 1.3)

# Add note about Triton's longer time
ax.annotate('Includes model conversion\nto TensorRT format',
            xy=(2, deploy_agg[deploy_agg['architecture']=='triton']['mean'].values[0]),
            xytext=(1.5, 400),
            fontsize=9,
            arrowprops=dict(arrowstyle='->', color='gray'),
            ha='center')

plt.tight_layout()

# Save PNG only (PDF generated via LaTeX separately)
plt.savefig(PLOTS_DIR / 'rq3_deployment_time.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: rq3_deployment_time.png")

# Deployment Time Summary Table with Mean and SD
print()
print("Deployment Time Summary (with Standard Deviation)")
print("=" * 60)
deploy_summary = deploy_agg.copy()
deploy_summary['architecture'] = [ARCH_DISPLAY_NAMES[a] for a in arch_order]
deploy_summary.columns = ['Architecture', 'Mean Time (s)', 'SD (s)']

# Add vs Monolithic ratio column
mono_mean = deploy_summary[deploy_summary['Architecture'] == 'Monolithic']['Mean Time (s)'].values[0]
deploy_summary['vs Monolithic'] = (deploy_summary['Mean Time (s)'] / mono_mean).apply(lambda x: f'{x:.1f}x')

display(deploy_summary)

# Save deployment summary
deploy_summary.to_csv(PLOTS_DIR / 'rq3_deployment_summary.csv', index=False)
print("\nSaved: rq3_deployment_summary.csv")

### H3c: Monolithic has shortest deployment time

In [ ]:
# H3c hypothesis validation
# Calculate mean deployment time per architecture
deploy_means = deployment_times.groupby('architecture')['total_time_seconds'].mean()

monolithic_time = deploy_means['monolithic']
microservices_time = deploy_means['microservices']
triton_time = deploy_means['triton']

print("H3c: Deployment Time Comparison")
print("=" * 50)
print(f"\n{'Architecture':<15} {'Mean Time (s)':>15} {'vs Monolithic':>15}")
print("-" * 47)
for arch in ['monolithic', 'microservices', 'triton']:
    time = deploy_means[arch]
    ratio = time / monolithic_time
    print(f"{ARCH_DISPLAY_NAMES[arch]:<15} {time:>15.1f} {ratio:>14.1f}x")

print()

# Testable prediction: monolithic.deployment_time < min(microservices, triton)
min_others = min(microservices_time, triton_time)
h3c_supported = monolithic_time < min_others

print(f"Testable Prediction: monolithic.time ({monolithic_time:.1f}s) < min(microservices, triton) ({min_others:.1f}s)")
print(f"Result: {monolithic_time:.1f} < {min_others:.1f} = {h3c_supported}")
print(f"\nH3c Supported: {h3c_supported}")

print(f"\nNote: Triton deployment (~{triton_time:.0f}s) is ~{triton_time/monolithic_time:.0f}x slower than Monolithic")
print(f"      due to ONNX-to-TensorRT model conversion during startup.")

# Cohen's d effect size for H3c deployment time comparisons
print()
print("=" * 50)
print("Effect Size Analysis (Cohen's d)")
print("=" * 50)

mono_times = deployment_times[deployment_times['architecture']=='monolithic']['total_time_seconds']
micro_times = deployment_times[deployment_times['architecture']=='microservices']['total_time_seconds']
triton_times = deployment_times[deployment_times['architecture']=='triton']['total_time_seconds']

h3c_d_micro = cohens_d(mono_times, micro_times)
h3c_d_triton = cohens_d(mono_times, triton_times)

print(f"\nMonolithic vs Microservices:")
print(f"  Cohen's d = {h3c_d_micro:.2f} ({interp_d(h3c_d_micro)} effect)")
print(f"\nMonolithic vs Triton:")
print(f"  Cohen's d = {h3c_d_triton:.2f} ({interp_d(h3c_d_triton)} effect)")

print(f"\nInterpretation: The deployment time difference between Monolithic and Triton")
print(f"shows a {interp_d(h3c_d_triton)} effect size, indicating practically significant difference.")

## 3. Cloud Cost Analysis

Using AWS and GCP pricing data to calculate cost-per-request across architectures.

In [ ]:
# Parse cloud pricing from calculator exports
# AWS: t3.medium (2 vCPU, 4 GB), us-east-2 (Ohio), On-Demand
# GCP: E2 Custom (2 vCPU, 4 GB), us-central1, On-Demand

# Parse AWS pricing (Chinese format, first row is title, skip it)
# Row 1: 預付成本, 每月費用, Total 12 months cost, 貨幣
# Row 2: 0, 30.37, 364.44, USD
aws_df = pd.read_csv(aws_path, encoding='utf-8-sig', skiprows=1, nrows=1)
aws_monthly = float(aws_df.iloc[0, 1])  # 每月費用 column = monthly cost
print(f"AWS t3.medium monthly cost: ${aws_monthly:.2f}")

# Parse GCP pricing (standard CSV with metadata rows at end)
# Find row where sku column contains "Total"
gcp_df = pd.read_csv(gcp_path)
total_mask = gcp_df['sku'].str.contains('Total', na=False)
gcp_monthly = float(gcp_df.loc[total_mask, 'total_price, USD'].values[0])
print(f"GCP E2 Custom monthly cost: ${gcp_monthly:.2f}")

# Convert to hourly rates (730 hours/month average)
HOURS_PER_MONTH = 730
AWS_HOURLY_PER_INSTANCE = aws_monthly / HOURS_PER_MONTH
GCP_HOURLY_PER_INSTANCE = gcp_monthly / HOURS_PER_MONTH

print(f"\nHourly rates:")
print(f"  AWS: ${AWS_HOURLY_PER_INSTANCE:.4f}/hour")
print(f"  GCP: ${GCP_HOURLY_PER_INSTANCE:.4f}/hour")

# Container counts per architecture (loaded from experiment.yaml via ResultsLoader)
CONTAINER_COUNTS = ResultsLoader.CONTAINER_COUNTS

print(f"\nContainer counts (from experiment.yaml):")
for arch, count in CONTAINER_COUNTS.items():
    print(f"  {arch}: {count} container(s)")

# Calculate hourly cost per architecture
cost_data = []
for arch in arch_order:
    containers = CONTAINER_COUNTS[arch]
    aws_hourly = AWS_HOURLY_PER_INSTANCE * containers
    gcp_hourly = GCP_HOURLY_PER_INSTANCE * containers
    cost_data.append({
        'Architecture': ARCH_DISPLAY_NAMES[arch],
        'Containers': containers,
        'AWS Hourly ($)': aws_hourly,
        'GCP Hourly ($)': gcp_hourly,
    })

hourly_costs = pd.DataFrame(cost_data)
print("\nHourly Costs by Architecture")
print("=" * 60)
display(hourly_costs)

In [ ]:
# Cost per request calculation
# Load throughput data dynamically from summary.csv at 100 concurrent users
# (instead of hardcoding values)
loader = ResultsLoader()
THROUGHPUT_RPS = loader.get_throughput_at_load(100)

print("Throughput values loaded from summary.csv (at 100 concurrent users):")
for arch, rps in THROUGHPUT_RPS.items():
    print(f"  {arch}: {rps:.2f} RPS")
print()

# Formula: cost_per_1000_requests = (hourly_cost / rps) * (1000 / 3600)
#        = hourly_cost * 1000 / (rps * 3600)
#        = hourly_cost / rps * 0.2778

cost_per_req_data = []
for arch in arch_order:
    containers = CONTAINER_COUNTS[arch]
    aws_hourly = AWS_HOURLY_PER_INSTANCE * containers
    gcp_hourly = GCP_HOURLY_PER_INSTANCE * containers
    rps = THROUGHPUT_RPS[arch]
    
    aws_per_1k = (aws_hourly / rps) * (1000 / 3600)
    gcp_per_1k = (gcp_hourly / rps) * (1000 / 3600)
    
    cost_per_req_data.append({
        'Architecture': ARCH_DISPLAY_NAMES[arch],
        'AWS Hourly ($)': aws_hourly,
        'GCP Hourly ($)': gcp_hourly,
        'RPS': rps,
        'AWS $/1K Req': aws_per_1k,
        'GCP $/1K Req': gcp_per_1k,
    })

cost_df = pd.DataFrame(cost_per_req_data)

print("Cost Per 1,000 Requests by Architecture")
print("=" * 70)
display(cost_df.round(5))

In [ ]:
# Cost efficiency ratios and validation
# Calculate ratios vs monolithic baseline
mono_aws_per_1k = cost_df[cost_df['Architecture'] == 'Monolithic']['AWS $/1K Req'].values[0]
mono_gcp_per_1k = cost_df[cost_df['Architecture'] == 'Monolithic']['GCP $/1K Req'].values[0]

triton_aws_per_1k = cost_df[cost_df['Architecture'] == 'Triton']['AWS $/1K Req'].values[0]
triton_gcp_per_1k = cost_df[cost_df['Architecture'] == 'Triton']['GCP $/1K Req'].values[0]

print("Cost Efficiency Ratios (vs Monolithic Baseline)")
print("=" * 60)
print(f"\n{'Architecture':<15} {'AWS Ratio':>12} {'GCP Ratio':>12}")
print("-" * 41)

for _, row in cost_df.iterrows():
    aws_ratio = row['AWS $/1K Req'] / mono_aws_per_1k
    gcp_ratio = row['GCP $/1K Req'] / mono_gcp_per_1k
    print(f"{row['Architecture']:<15} {aws_ratio:>11.2f}x {gcp_ratio:>11.2f}x")

# Validate key finding: Triton ~2x more expensive than Monolithic
triton_aws_ratio = triton_aws_per_1k / mono_aws_per_1k
triton_gcp_ratio = triton_gcp_per_1k / mono_gcp_per_1k

print("\n" + "=" * 60)
print("KEY FINDING VALIDATION")
print("=" * 60)
print(f"\nTriton/Monolithic cost ratio:")
print(f"  AWS: {triton_aws_ratio:.2f}x")
print(f"  GCP: {triton_gcp_ratio:.2f}x")

# Assert ratio is approximately 2x (allow 1.9-2.1 range for floating point)
assert 1.9 < triton_aws_ratio < 2.1, f"AWS ratio {triton_aws_ratio:.2f} not in expected range [1.9, 2.1]"
assert 1.9 < triton_gcp_ratio < 2.1, f"GCP ratio {triton_gcp_ratio:.2f} not in expected range [1.9, 2.1]"

print(f"\nValidated: Monolithic is ~2x cheaper than Triton per request")
print(f"  (Ratios within expected range [1.9, 2.1])")

In [ ]:
# Cost comparison grouped bar chart
fig, ax = plt.subplots(figsize=(10, 6))

x_pos = np.arange(len(arch_order))
bar_width = 0.35

# AWS bars
aws_costs = cost_df['AWS $/1K Req'].values
bars1 = ax.bar(x_pos - bar_width/2, aws_costs * 1000, bar_width, 
               label='AWS', color='#FF9900', edgecolor='black')

# GCP bars
gcp_costs = cost_df['GCP $/1K Req'].values
bars2 = ax.bar(x_pos + bar_width/2, gcp_costs * 1000, bar_width, 
               label='GCP', color='#4285F4', edgecolor='black')

# Add value labels on bars
for bars, costs in [(bars1, aws_costs), (bars2, gcp_costs)]:
    for bar, cost in zip(bars, costs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'${cost:.4f}', ha='center', va='bottom', fontsize=9)

# Customize axes
ax.set_xticks(x_pos)
ax.set_xticklabels([ARCH_DISPLAY_NAMES[a] for a in arch_order])
ax.set_ylabel('Cost per 1,000 Requests (USD, x10$^{-3}$)')
ax.set_title('Cost per 1,000 Requests by Architecture and Cloud Provider')
ax.legend()
ax.set_ylim(0, max(max(aws_costs), max(gcp_costs)) * 1000 * 1.3)

plt.tight_layout()

# Save in both formats
plt.savefig(PLOTS_DIR / 'rq3_cost_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: rq3_cost_comparison.png")

## 4. Hypothesis Results Summary

In [ ]:
# Summary table of all H3 hypothesis results with effect sizes
# Re-extract effect sizes for deployment time (H3c)
mono_times = deployment_times[deployment_times['architecture']=='monolithic']['total_time_seconds']
micro_times = deployment_times[deployment_times['architecture']=='microservices']['total_time_seconds']
triton_times = deployment_times[deployment_times['architecture']=='triton']['total_time_seconds']

h3c_d_micro = cohens_d(mono_times, micro_times)
h3c_d_triton = cohens_d(mono_times, triton_times)

# Use the larger effect size (Triton comparison) for H3c since it's the key finding
h3c_effect_size = h3c_d_triton

hypothesis_results = pd.DataFrame([
    {
        'Hypothesis': 'H3a',
        'Statement': 'Triton requires fewest application LOC',
        'Predicted': f'triton < monolithic',
        'Observed': f'{triton_app_loc} vs {monolithic_app_loc}',
        'Supported': h3a_supported,
        'Effect_Size': 'N/A',
        'Interpretation': 'single measurement'
    },
    {
        'Hypothesis': 'H3b',
        'Statement': 'Microservices requires most configuration LOC',
        'Predicted': f'micro > max(mono, triton)',
        'Observed': f'{microservices_config_loc} vs {max_others}',
        'Supported': h3b_supported,
        'Effect_Size': 'N/A',
        'Interpretation': 'single measurement'
    },
    {
        'Hypothesis': 'H3c',
        'Statement': 'Monolithic has shortest deployment time',
        'Predicted': f'mono < min(micro, triton)',
        'Observed': f'{monolithic_time:.0f}s vs {min_others:.0f}s',
        'Supported': h3c_supported,
        'Effect_Size': f'd={h3c_effect_size:.2f}',
        'Interpretation': interp_d(h3c_effect_size)
    },
])

print("\nRQ3 Hypothesis Results (with Effect Sizes)")
print("=" * 90)
display(hypothesis_results)

# Save to CSV
hypothesis_results.to_csv(PLOTS_DIR / 'rq3_hypothesis_results.csv', index=False)
print("\nSaved: rq3_hypothesis_results.csv")

## 5. Key Findings

**Lines of Code:**
- Microservices has highest total LOC (752), followed by Triton (524), then Monolithic (374)
- H3a NOT supported: Triton (439 LOC) requires MORE application code than Monolithic (317 LOC) due to gateway service
- H3b supported: Microservices (89 LOC) has most configuration, exceeding Triton (85 LOC) and Monolithic (57 LOC)

**Deployment Time:**
- H3c supported: Monolithic (~57s) deploys fastest
- Triton (~600s) is ~10x slower due to ONNX-to-TensorRT model conversion at startup
- Microservices (~79s) has modest overhead from multi-container orchestration

**Cost Analysis:**
- Monolithic is ~2x cheaper per request than Triton (consistent across AWS and GCP)
- Microservices is ~1.6x more expensive than Monolithic (higher throughput partially offsets doubled resources)
- At 100M daily requests: Monolithic saves ~$37K/year (AWS) or ~$53K/year (GCP) vs Triton